# NASA FIRMS full-history access test

This notebook tests a remote query over every yearly partition in the public Source Cooperative FIRMS GeoParquet catalog. It aggregates detections by year, sensor, and quality; it does **not** download all observation rows to this computer. The catalog is a mirror of NASA FIRMS, and recent observations may later be replaced by science-quality data.

In [ ]:
import duckdb

print(f"DuckDB {duckdb.__version__}")
con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("SET s3_region = 'us-west-2'")
print("DuckDB httpfs loaded; ready to query the public S3 catalog.")

In [ ]:
catalog = (
    "s3://us-west-2.opendata.source.coop/"
    "portolan-mirrors/firms-catalog/detections/"
    "year=*/detections.parquet"
)

query = """
    SELECT year, sensor, quality, count(*) AS detections
    FROM read_parquet(?, hive_partitioning = true)
    GROUP BY year, sensor, quality
    ORDER BY year, sensor, quality
"""

import csv

result = con.execute(query, [catalog])
columns = [column[0] for column in result.description]
rows = result.fetchall()
if not rows:
    raise RuntimeError("The catalog query returned no detections.")

year_totals = {}
for year, sensor, quality, detections in rows:
    year_totals[year] = year_totals.get(year, 0) + detections

print(f"Years returned: {min(year_totals)}–{max(year_totals)}")
print(f"Total detections in queried partitions: {sum(year_totals.values()):,}")
print(f"Year partitions with detections: {len(year_totals)}")
summary_csv = "firms_detection_summary_by_year_sensor_quality.csv"
with open(summary_csv, "w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(columns)
    writer.writerows(rows)

print(f"Saved {len(rows)} summary rows to: {summary_csv}")


## Optional: export every raw detection

The summary above contains only grouped counts. To export the individual detection records instead, run the cell below. It streams the query through DuckDB rather than loading all records into a pandas DataFrame, but the full-history CSV can be very large and take a long time to create. Ensure you have enough disk space first.

In [ ]:
raw_csv = "firms_all_detections.csv"
catalog_sql = catalog.replace("'", "''")
output_sql = raw_csv.replace("'", "''")

con.execute(f"""
    COPY (
        SELECT *
        FROM read_parquet('{catalog_sql}', hive_partitioning = true)
    )
    TO '{output_sql}' (HEADER, DELIMITER ',')
""")
print(f"Saved all raw detections to: {raw_csv}")